# Slot 2: Build a CNN that predicts AGO2 binding

**Technion AI Days 2026: Teaching Machines to Read DNA**

**AGO2** is a protein that binds RNA. CLIP-seq is an experiment that crosslinks AGO2 to the RNA it
is bound to and sequences those RNA fragments. From such data we have thousands of RNA sequences
that AGO2 was bound to (**positives**) and sequences it was not found bound to (**negatives**).
In the next 30 minutes you will build a model that takes an RNA sequence and estimates the
probability that AGO2 binds it:

$$f(\text{sequence}) = P(\text{bound} \mid \text{sequence})$$

**How this notebook works**

| cell | what you do |
|---|---|
| `#@title ... run this cell, nothing to change` | run it (Shift+Enter) |
| `# TODO-N` with `# === Your code here ===` | write the code yourself (replace the `...`) |

Before you start: *Runtime → Change runtime type → T4 GPU*.

In [1]:
#@title Setup - run this cell, nothing to change { display-mode: "form" }
import os, time, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn, optim
from sklearn.metrics import roc_auc_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU - Runtime > Change runtime type > T4 GPU")

# Plot style, shared by every notebook (the same palette as the slides).
import logging
import matplotlib.pyplot as plt
from matplotlib import cycler
from matplotlib.colors import LinearSegmentedColormap

INK, MUTED, MIDGREY, LIGHTGREY, HAIRLINE = "#202124", "#5F6368", "#80868B", "#C4C7C5", "#DADCE0"
ACCENT = "#1A73E8"
BOUND, UNBOUND = ACCENT, LIGHTGREY                            # class colours
CNN_C, FROZEN_C, FINETUNED_C, BASELINE_C = INK, MIDGREY, ACCENT, MIDGREY   # model colours
BASE_COLOURS = {"A": "#0F9D58", "C": "#4285F4", "G": "#F4B400", "U": "#DB4437", "T": "#DB4437"}  # logos only
CMAP_DIV = LinearSegmentedColormap.from_list("ws_div", [MUTED, "#FFFFFF", ACCENT])         # weight heatmaps
CMAP_SEQ = LinearSegmentedColormap.from_list("ws_seq", ["#E8F0FE", ACCENT, "#174EA6"])     # sequential values

plt.colormaps.register(CMAP_SEQ, force=True)
plt.colormaps.register(CMAP_DIV, force=True)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # Colab has no Roboto: fall back quietly
plt.rcParams.update({
    "font.family": ["Roboto", "DejaVu Sans"],
    "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "axes.titlesize": 11,
    "axes.prop_cycle": cycler(color=[ACCENT, INK, MIDGREY, LIGHTGREY]),
    "axes.grid": False, "grid.color": HAIRLINE, "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 8,
    "image.cmap": "ws_seq",
})

# Where the workshop files live. Either one direct-download link per file (FILE_URLS, e.g. SharePoint
# "anyone with the link" + download=1), or one folder URL (DATA_URL, e.g. a Hugging Face dataset repo).
FILE_URLS = {}
DATA_URL = os.environ.get("AIDAYS_DATA_URL", "https://huggingface.co/datasets/yup8/ai-days-2026-ago2/resolve/main")
DATA_DIR = Path(os.environ.get("AIDAYS_DATA_DIR", "data"))
DATA_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def fetch(name):
    """Return the local path of a workshop file, downloading it first if needed."""
    path = DATA_DIR / name
    if not path.exists():
        url = FILE_URLS.get(name) or (f"{DATA_URL.rstrip('/')}/{name}" if DATA_URL else "")
        if not url:
            raise FileNotFoundError(f"{path} not found and no download link is set for it")
        t = time.time()
        urllib.request.urlretrieve(url, path)
        print(f"downloaded {name} ({path.stat().st_size / 1e6:.1f} MB) in {time.time() - t:.1f}s")
    return path


def download_outputs(*names):
    """End of a notebook: save your results to your laptop so the next notebook can use them."""
    for n in names:
        p = DATA_DIR / n
        if not p.exists():
            print(f"{n}: not found (did the cell that makes it run?)")
        elif IN_COLAB:
            _colab_files.download(str(p))
        else:
            print(f"{n}: saved at {p}")


def upload_yours():
    """Start of a notebook: upload the files you downloaded at the end of an earlier notebook."""
    if not IN_COLAB:
        print(f"not in Colab: copy your files into {DATA_DIR}")
        return
    import re
    for n, b in _colab_files.upload().items():
        clean = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", n)   # "my_cnn (1).pt" -> "my_cnn.pt" (browser renames)
        (DATA_DIR / clean).write_bytes(b)
        print("uploaded", n, "->", clean)

device: cuda | NVIDIA A30


In [2]:
#@title Data helpers - run this cell, nothing to change { display-mode: "form" }
MAX_LEN  = 501          # fixed global length (pad / truncate)
ALPHABET = "ACGU"       # order = the 4 rows of the one-hot matrix
FILES = {split: (f"CLIPSEQ_AGO2.{split}.positives.fa", f"CLIPSEQ_AGO2.{split}.negatives.fa")
         for split in ("train", "ls")}


def read_fasta(path, label):
    """Return (sequences, labels) from a FASTA file, all with the same label.

    The files mix the DNA letter T and the RNA letter U for the same base,
    so we normalise everything to U.
    """
    seqs, labs = [], []
    with Path(path).open() as fh:
        for line in fh:
            if line.startswith(">"):
                continue
            seqs.append(line.strip().upper().replace("T", "U"))
            labs.append(label)
    return seqs, labs


def read_split(split):
    """'train' or 'ls' (the held-out validation set) -> (sequences, labels as a numpy array)."""
    pos, neg = FILES[split]
    p, lp = read_fasta(fetch(pos), 1)
    n, ln = read_fasta(fetch(neg), 0)
    return p + n, np.array(lp + ln)


_LUT = np.full(256, -1, dtype=np.int64)
for _i, _b in enumerate(ALPHABET):
    _LUT[ord(_b)] = _i


def encode_batch(seqs, max_len=MAX_LEN):
    """List of RNA strings -> one-hot FloatTensor (B, 1, 4, max_len).

    Pads/truncates to max_len; padding and unknown bases stay uniform 0.25.
    """
    X = np.full((len(seqs), 1, 4, max_len), 0.25, dtype=np.float32)
    for b, s in enumerate(seqs):
        idx = _LUT[np.frombuffer(s[:max_len].encode(), dtype=np.uint8)]
        pos = np.nonzero(idx >= 0)[0]
        X[b, 0, :, pos] = 0.0
        X[b, 0, idx[pos], pos] = 1.0
    return torch.from_numpy(X)

## 1. The data

Each sequence is a string over the 4-letter RNA alphabet `A C G U`. The *train* files are used
to fit the model; the *ls* files are a separate held-out set that we use only for evaluation.

In [3]:
#@title Load the sequences - run this cell, nothing to change { display-mode: "form" }
train_seqs, train_y = read_split("train")
val_seqs, val_y = read_split("ls")
print(f"train: {len(train_seqs):,} sequences ({train_y.mean():.0%} bound)")
print(f"ls   : {len(val_seqs):,} sequences ({val_y.mean():.0%} bound)")
print(f"length: {min(map(len, train_seqs))}-{max(map(len, train_seqs))} letters")
print("\na bound sequence:\n", train_seqs[0][:90], "...")

train: 92,346 sequences (52% bound)
ls   : 1,000 sequences (50% bound)
length: 137-375 letters

a bound sequence:
 GGACAGAAGGGUAUAGAAAAGUCCCCUACCAUUAUUAUGAACAAGGAAGAGAUGAGUGUGAUGAAUAUUUUCUUCAUGAACAUGCCCCAU ...


## 2. One-hot encoding: a sequence becomes a matrix

The model takes numbers as input. We encode each letter as a column of 4 numbers: 1 in the row of
that letter and 0 elsewhere (rows = `A, C, G, U`). A sequence of length $L$ becomes a $4 \times L$ matrix,
which we treat as a one-channel image of height 4. Sequences shorter than 501 letters are padded
to 501 with columns of 0.25 (equal weight on all four letters).

In [4]:
# TODO-1: one-hot encode the FIRST training sequence with `encode_batch` and print
#         its first 8 columns. Check them against the first 8 letters printed above.
#         `encode_batch` takes a list of strings and returns shape (N, 1, 4, 501).
# === Your code here ===
...
# === End of your code ===

Ellipsis

## 3. The model

A **convolution** layer computes the dot product of a small weight matrix (a *filter*) with every
window of the input. Our first-layer filters are $4 \times 10$, so each one gives a score to every
10-letter window. The score is high when the window matches the letter pattern stored in the
filter's weights, so each filter acts as a detector for one **motif** (a short sequence pattern).
Max pooling keeps the largest value in each block of 3 positions. A second convolution combines the
first-layer outputs, and a final linear layer maps them to one number per sequence, the *logit*.

Shapes, for one sequence of width 501:

| layer | output shape |
|---|---|
| input | `1 × 4 × 501` |
| `Conv2d(1, 10, (4, 10))` + ReLU | `10 × 1 × 492` |
| `MaxPool2d((1, 3))` | `10 × 1 × 164` |
| `Conv2d(10, 20, (1, 5))` + ReLU | `20 × 1 × 160` |
| `MaxPool2d((1, 3))` | `20 × 1 × 53` |
| `Flatten` → `Linear(20*53, 1)` | `1` |

In [5]:
class GlobalCNN(nn.Module):
    """One-hot batch (N, 1, 4, 501) -> one raw score (logit) per sequence, shape (N,)."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # TODO-2: build the network from the table above, in order, separated by commas.
            #         The last layer is nn.Linear(20 * 53, 1).
            # === Your code here ===
            ...
            # === End of your code ===
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


# sanity check: two sequences in, two scores out
print(GlobalCNN()(encode_batch(train_seqs[:2])).shape, "<- should be torch.Size([2])")

TypeError: ellipsis is not a Module subclass

## 4. Training

Training minimises a **loss** by gradient descent (here the Adam variant): each step computes the
gradient of the loss on one batch of 256 sequences and moves every weight a small amount against
its gradient. Our loss is binary cross-entropy, the negative log of the probability the model
assigns to the correct label, averaged over the batch. The model outputs a logit $z$; the sigmoid
$\sigma(z) = 1 / (1 + e^{-z})$ (`torch.sigmoid`) turns it into a probability between 0 and 1.

One training step is always the same four lines (find them in the cell below):

| line | what it does |
|---|---|
| `opt.zero_grad()` | reset the gradients from the previous step to zero |
| `loss = criterion(model(X), y)` | compute the loss on one batch |
| `loss.backward()` | compute the gradient of the loss for every weight (backpropagation) |
| `opt.step()` | update every weight using its gradient |

After every pass over the training data (an *epoch*) we evaluate the model on the held-out test
set: the 1,000 `ls` sequences it never trains on. `model.eval()` puts the model in evaluation mode
and `torch.no_grad()` turns off gradient tracking, so scoring these sequences does not change the
weights. The number we report
is **AUROC**: the probability that a randomly chosen bound sequence gets a higher score than a
randomly chosen unbound one. 0.5 is a coin flip, 1.0 is perfect.

In [6]:
def train(model, train_loader, val_seqs, val_y, epochs=10, lr=1e-3, wd=1e-4):
    model.to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.BCEWithLogitsLoss()
    X_val = encode_batch(val_seqs)
    history = []
    for ep in range(1, epochs + 1):
        t = time.time()
        model.train()
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            opt.step()

        # TODO-3: measure the model on the held-out set.
        #         1. switch to eval mode and turn off gradients (torch.no_grad())
        #         2. probabilities = sigmoid of the model's output on X_val (move X_val to DEVICE)
        #         3. auc = roc_auc_score(true labels, probabilities)
        # === Your code here ===
        ...
        # === End of your code ===
        history.append(auc)
        print(f"epoch {ep:02d} | held-out AUROC = {auc:.4f} | {time.time() - t:.1f}s")
    return history

In [7]:
#@title Encode everything and train - run this cell, nothing to change { display-mode: "form" }
from torch.utils.data import DataLoader, TensorDataset

t0 = time.time()
X_train = encode_batch(train_seqs)
y_train = torch.tensor(train_y, dtype=torch.float32)
print(f"encoded {len(train_seqs):,} sequences in {time.time() - t0:.1f}s")
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=256, shuffle=True, drop_last=True)

EPOCHS = 10
torch.manual_seed(SEED)
model = GlobalCNN()
history = train(model, train_loader, val_seqs, val_y, epochs=EPOCHS)
torch.save(model.state_dict(), DATA_DIR / "my_cnn.pt")
print(f"\nfinal held-out AUROC: {history[-1]:.3f} - saved to {DATA_DIR / 'my_cnn.pt'}")

encoded 92,346 sequences in 1.5s


TypeError: ellipsis is not a Module subclass

In [8]:
#@title Plot the learning curve - run this cell, nothing to change { display-mode: "form" }
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.plot(range(1, len(history) + 1), history, marker="o", c=CNN_C, label="Your CNN")
plt.axhline(0.5, ls="--", c=MIDGREY, lw=1, label="Coin flip")
plt.xlabel("Epoch"); plt.ylabel("Held-out AUROC"); plt.ylim(0.45, 1); plt.legend(); plt.grid(axis="y")
plt.title("Learning curve"); plt.tight_layout(); plt.show()

NameError: name 'history' is not defined

<Figure size 500x300 with 0 Axes>

**Reading your result.** The curve should climb quickly and level off around **0.86**. Two training
runs differ by less than 0.01. The test set has only 1,000 sequences, so any AUROC on it has a
sampling uncertainty of about ±0.025 (Slot 1). A 0.85 and a 0.86 are the same result; a 0.80 and a
0.86 are not.

## 5. If you finish early

Change `EPOCHS` (e.g. 3 or 20) and rerun the training cell. Does more training still help after
epoch 10? Compare with your neighbour. (Keep the architecture as it is: Slot 3 loads this exact shape.)

## 6. Take your model with you

Each Colab notebook runs on its own machine. Download your trained CNN now. At the start of Slot 3
you can upload it, or use ours.

In [9]:
#@title Download my_cnn.pt - run this cell { display-mode: "form" }
download_outputs("my_cnn.pt")

my_cnn.pt: not found (did the cell that makes it run?)
